In [1]:
# Description:
#     This script creates the json files to be used in the vizTool
#
#     Date Edited: 08/20/2024


# ============================================================================================================
# System setup
# ============================================================================================================
print("\nRunning Python Script: 'vt_CompileJsons.py'")
print("\n\nSystem setup...")
print("\n    load python libraries")

import sys, os, time, traceback
time_begin = time.time()
time_begin_SysSetup = time.time()
import pandas as pd
import importlib.machinery
import json
import vizTool_Scripts as scripts

# calculate elapsed time
time_end_SysSetup = time.time()
scripts.printElapsedTime(time_end_SysSetup, time_begin_SysSetup)


try:
    
    # ============================================================================================================
    # Global Parameters
    # ============================================================================================================
    print("\n\nSpecifying Global Parameters & Input-Output Files...")
    
    # get time (in seconds) 
    time_begin_GlobalParam = time.time()
    
    # ----------------------------------------------------------------------------------------
    # specify global parameters
    # ----------------------------------------------------------------------------------------
    in_GlobalVar_txt     = "py_Variables - vt_JsonVars.txt"
    out_Log_txt_base     = "py_LogFile - vt_CompileJson"
    
    # create global variables from input file
    #dir_ScriptLaunch = os.getcwd()
    dir_ScriptLaunch = r'D:\GitHub\WF-TDM-v9x\Scenarios\WF TDM v910-Task2\BY_2019'
    
    # create path to global variables input file
    path_in_GlobalVar_txt = os.path.join(dir_ScriptLaunch, "_Log", in_GlobalVar_txt)
    
    # create variables from input file global variables
    GlobalVars = importlib.machinery.SourceFileLoader('data', path_in_GlobalVar_txt).load_module()

    # create global parameters
    jsonId        = GlobalVars.jsonId
    ParentDir     = GlobalVars.ParentDir
    ScenarioDir   = GlobalVars.ScenarioDir
    ModelVersion  = GlobalVars.ModelVersion
    ScenarioGroup = GlobalVars.ScenarioGroup
    RunYear       = GlobalVars.RunYear
    
    # specifiy inputs
    print("\n    define input/output locations")
    input_file = GlobalVars.inputFile
    
    # define scenario
    _scenarioCode = ModelVersion + '__' + ScenarioGroup + '__' + str(RunYear)

    # define output path
    output_path = os.path.join(ParentDir, r"Scenarios\\.vizTool\\scenario-data\\" + _scenarioCode + "\\")
    
    # Ensure output path exists
    if not os.path.exists(output_path):
        os.makedirs(output_path)
    
    #define vizTool scripts path
    vizscripts_path = ParentDir + "2_ModelScripts\_Python\py-vizTool"
    
    # ----------------------------------------------------------------------------------------
    # begin log file
    # ----------------------------------------------------------------------------------------
    out_Log_txt = out_Log_txt_base + jsonId + '.txt'
    path_out_Log_txt = os.path.join(ParentDir, ScenarioDir, "_Log", out_Log_txt)
    logFile = open(path_out_Log_txt, 'w')
    
    # calculate elapsed time
    logFile.write("\nRunning Python Script: 'vt_CompileJson.py'")
    logFile.write("\n\nSystem setup...")
    logFile.write("\n    load python libraries")
    scripts.logElapsedTime(time_end_SysSetup, time_begin_SysSetup, logFile)
    
    logFile.write("\n\nSpecifying Global Parameters & Input-Output Files...")
    logFile.write("\n    define input/output locations")
    
    time_end_GlobalParam = time.time()
    scripts.printElapsedTime(time_end_GlobalParam, time_begin_GlobalParam)
    scripts.logElapsedTime(time_end_GlobalParam, time_begin_GlobalParam, logFile)
    
    
    # ============================================================================================================
    # Config Parameters and Other Inputs
    # ============================================================================================================
    print("\n\nSpecifying vizTool Config Parameters and Inputs...")
    print("\n    initialize config.json file")
    logFile.write("\n\nSpecifying vizTool Config Parameters and Inputs...")
    logFile.write("\n    initialize config.json file")
    
    # get time (in seconds) 
    time_begin_ConfigParam = time.time()

    # read in viztool config file and get json variables
    with open(vizscripts_path + "\config.json") as file:
        config_data = json.load(file)

    # read output file name & create path for it
    output_file_name = config_data.get(jsonId, {}).get("output_file_name")
    output_file = os.path.join(output_path, output_file_name)
    logFile.write("\n\nOUTPUT_FILE" + output_file)
    
    # define config parameters
    print("\n    read in config params")
    logFile.write("\n    read in config params")
    
    group_and_sum          = config_data.get(jsonId, {}).get("group_and_sum", False)           # aggregate by col column values
    some_atts_have_dupdata = config_data.get(jsonId, {}).get("some_atts_have_dupdata", False)  # when duplication of data in rows, duplicates for repetive column keys are removed
    filter_col             = config_data.get(jsonId, {}).get("filter_col", "")                 # column to filter
    filter_val             = config_data.get(jsonId, {}).get("filter_val", "")                 # value to filter 
    id_col                 = config_data.get(jsonId, {}).get("id_col")                         #
    
    # ----------------------------------------------------------------------------------------
    # Set attributes and filters 
    # ----------------------------------------------------------------------------------------
    attributes           = pd.read_csv(vizscripts_path + '\\attributes.csv')
    filters              = pd.read_csv(vizscripts_path + '\\filters.csv')
    colKeysAndMeltParams = pd.read_csv(vizscripts_path + '\\col-keys-and-melt-params.csv')

    # attributes
    dfAttributes = attributes.query('ID == @jsonId').drop(columns=['Step', 'Script', 'ID'])

    # convert string of list to actual list
    dfAttributes['keyColsForDupDataRemoval'] = dfAttributes['keyColsForDupDataRemoval'].apply(
        lambda x: [item.strip() for item in x.strip("[]").split(",")] 
        if pd.notna(x) and isinstance(x, str) and x.lower() not in ['', 'na'] 
        else []
    )

    # filters
    dfFilters    = filters.query('ID == @jsonId').drop(columns=['Step', 'Script', 'ID'])
    dfFilters.fillna("", inplace=True)

    # col keys
    dfColKeysAndMeltParams = colKeysAndMeltParams.query('ID == @jsonId').drop(columns=['Step', 'Script', 'ID'])
    dfColKeysAndMeltParams.dropna(axis=1, thresh=1, inplace=True)
    dfColKeysAndMeltParams = dfColKeysAndMeltParams.where(pd.notna(dfColKeysAndMeltParams), None)
    
    time_end_ConfigParam = time.time()
    scripts.printElapsedTime(time_end_ConfigParam, time_begin_ConfigParam)
    scripts.logElapsedTime(time_end_ConfigParam, time_begin_ConfigParam, logFile)
    
    
    # ============================================================================================================
    # Reorganize Json Data
    # ============================================================================================================
    print("\n\nReorganizing Json Data...")
    print("\n    run reorganize_df_for_json function")
    logFile.write("\n\nReorganizing Json Data...")
    logFile.write("\n    run reorganize_df_for_json function")
    
    # get time (in seconds) 
    time_begin_Reorganize = time.time()

    df_data_with_keys = scripts.reorganize_df_for_json(
        dfColKeysAndMeltParams,
        dfFilters,
        dfAttributes,
        input_file,
        id_col,
        group_and_sum,
        some_atts_have_dupdata,
        filter_col,
        filter_val
    )
    
    time_end_Reorganize = time.time()
    scripts.printElapsedTime(time_end_Reorganize, time_begin_Reorganize)
    scripts.logElapsedTime(time_end_Reorganize, time_begin_Reorganize, logFile)


    # ============================================================================================================
    # Generate Json Data
    # ============================================================================================================
    print("\n\Generate Json Data...")
    print("\n    run generate_scenario_json function")
    logFile.write("\n\Generate Json Data...")
    logFile.write("\n    run generate_scenario_json function")
    
    # get time (in seconds) 
    time_begin_Generate = time.time()

    scripts.generate_scenario_json(
        df_data_with_keys,
        dfFilters,
        dfAttributes,
        _scenarioCode,
        output_file,
        id_col
    )
    
    time_end_Generate = time.time()
    scripts.printElapsedTime(time_end_Generate, time_begin_Generate)
    scripts.logElapsedTime(time_end_Generate, time_begin_Generate, logFile)
    logFile.close()



except:
    
    # print error message to log file if something doesn't work
    tb = sys.exc_info()[2]
    msg_error_traceback  = traceback.format_tb(tb)[0]
    msg_error_system     = str(sys.exc_info())
    msg_error_GeoPandas  = str(sys.exc_info()[1])
    
    logFile.write("\n")
    logFile.write("\n")
    logFile.write("\n")
    logFile.write(" ============================================================================================================\n")
    logFile.write("There was an error running this script...\n")
    logFile.write("\n")
    logFile.write("\n")
    logFile.write("PYTHON ERRORS:\n")
    logFile.write("\n")
    logFile.write("    Traceback info:\n")
    logFile.write("\n")
    logFile.write(msg_error_traceback + "\n")
    logFile.write("\n")
    logFile.write("\n")
    logFile.write("    Error Info:\n")
    logFile.write("\n")
    logFile.write(msg_error_system + "\n")
    logFile.write("\n")
    
    
    # print to console
    print("")
    print("")
    print("")
    print(" ============================================================================================================")
    print("*** There was an error running this script")
    print("Please check '" + ScenarioDir + "\\_Log\\" + out_Log_txt + "' for error messages.")
    print("")
    print("")
    
    
    # pause console to check printed messages
    input("Press Enter to continue...")
    
    
    # exit python & hand control back to Cube
    sys.exit(1)



Running Python Script: 'vizTool_CompileJsons.py'


System setup...

    load python libraries


ModuleNotFoundError: No module named 'vizTool_Scripts'